# Level 1: Tokenization and Data Preparation

We start by defining our training text and creating a character-level mapping (Tokenizer).

In [ ]:
# Sample text to train our "AI" on
text = "Hello world! This is a simple NLP entry-level LLM. It generates text character by character. Learning is fun."

# 1. Create the vocabulary (all unique characters)
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Vocabulary Size: {vocab_size}")
print(f"Unique characters: {''.join(chars)}")

# 2. Create a mapping from characters to integers (and vice versa)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

# 3. Define the Encoder and Decoder
encode = lambda s: [stoi[c] for c in s] # string to list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # list of integers to string

# Test it out
encoded_text = encode("Hello")
print(f"Encoded 'Hello': {encoded_text}")
print(f"Decoded back: {decode(encoded_text)}")

# Level 2: Model Architecture

We define the structure of our Small Language Model (SLM) using a Transformer-based architecture.

In [ ]:
import torch
import torch.nn as nn

class EntryLLM(nn.Module):
    def __init__(self, vocab_size, embed_size, num_layers):
        super().__init__()
        # 1. Turn text into numbers
        self.embedding = nn.Embedding(vocab_size, embed_size)
        
        # 2. The "Brain" (Transformer Blocks)
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_size, nhead=8, batch_first=True)
        self.blocks = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # 3. The "Output" (Predict the next word)
        self.output_layer = nn.Linear(embed_size, vocab_size)

    def forward(self, x):
        # x shape: (batch_size, context_length)
        x = self.embedding(x) # (batch_size, context_length, embed_size)
        x = self.blocks(x)    # (batch_size, context_length, embed_size)
        return self.output_layer(x) # (batch_size, context_length, vocab_size)

# Level 3: Data Loading and Batching

Prepare the data for training by splitting it into small chunks (batches).

In [ ]:
# Convert our text into a large tensor of integers
data = torch.tensor(encode(text), dtype=torch.long)

def get_batch(data, block_size, batch_size):
    # Generate random starting points in the data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    
    # Stack the inputs (X) and the targets (Y)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    
    return x, y

# Example check
xb, yb = get_batch(data, block_size=8, batch_size=4)
print(f"Input shape: {xb.shape}")
print(f"Target shape: {yb.shape}")

# Level 4: Training Pipeline

Run the optimization loop to teach the model how to predict the next character.

In [ ]:
import torch.optim as optim

# Hyperparameters
block_size = 16
batch_size = 32
learning_rate = 1e-3
training_steps = 1000

# 1. Initialize model and optimizer
model = EntryLLM(vocab_size=vocab_size, embed_size=128, num_layers=4)
optimizer = optim.AdamW(model.parameters(), lr=learning_rate)

print("Starting training...")

for steps in range(training_steps):
    xb, yb = get_batch(data, block_size, batch_size)
    
    logits = model(xb)
    
    # Reshape for loss calculation
    B, T, C = logits.shape
    logits = logits.view(B*T, C)
    targets = yb.view(B*T)
    
    loss = nn.functional.cross_entropy(logits, targets)
    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
    if steps % 200 == 0:
        print(f"Step {steps}: Loss {loss.item():.4f}")

print("Training complete.")

# Level 5: Inference (Text Generation)

Now that the model is trained, let's use it to generate some text!

In [ ]:
def generate(model, start_str, max_new_tokens=50):
    model.eval()
    # Encode the starting string
    context = torch.tensor([encode(start_str)], dtype=torch.long)
    
    for _ in range(max_new_tokens):
        # Crop context to block_size if necessary
        input_cond = context[:, -block_size:]
        
        # Get predictions
        logits = model(input_cond)
        logits = logits[:, -1, :]
        
        # Sample from the distribution
        probs = nn.functional.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        
        # Append to context
        context = torch.cat((context, next_token), dim=1)
        
    return decode(context[0].tolist())

print("Generating sample text:")
print(generate(model, start_str="Hello", max_new_tokens=40))